# Real-image self-calibration on DROID (no robot needed)
**GPU: T4 or L4.** Estimates each exterior camera's pose from the Franka's own body in REAL DROID frames and scores it
against DROID's post-hoc calibrated extrinsics (`KarlP/droid`, ungated). The demo's own arm motion is the sweep.

1. Export (~20 min, CPU): droid_100 episodes that have calibrated extrinsics; 8 frames per camera with joint angles.
2. Segment (~10 min): text-prompted robot masks (OWLv2 box -> SAM).
3. Calibrate (~1 to 2 min per camera): first checks the extrinsics convention by rendering the robot at the reference
   pose (paste that line), then grid + CEM over the 6-DoF camera pose.
4. Report: error vs the reference, and how often the estimate explains the silhouette as well as the reference.
The reference extrinsics are themselves estimates, so this measures agreement with DROID's calibration, not truth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline matplotlib backend crashes every venv python
os.environ['PYTHONUNBUFFERED'] = '1'   # run cells are piped into a log on Drive; without this their output arrives in bursts
os.makedirs('/content/drive/MyDrive/restoring_organs/plus/logs', exist_ok=True)
os.makedirs('/content/logs', exist_ok=True)
print('outputs ->', '/content/drive/MyDrive/restoring_organs/plus')
# Drive uploads a file only after it is CLOSED, so a log held open by a running cell never arrives. Run cells log to
# /content/logs; this thread rewrites the last 64 KB of each log on Drive once a minute as a fresh, closed file.
import builtins, glob, threading, time
def _mirror_logs():
    while True:
        for p in glob.glob('/content/logs/*.log'):
            try:
                with open(p, 'rb') as f:
                    f.seek(max(0, os.path.getsize(p) - 65536))
                    tail = f.read()
                with open('/content/drive/MyDrive/restoring_organs/plus/logs/' + os.path.basename(p)[:-4] + '.' + os.uname().nodename[-6:] + '.log', 'wb') as f:
                    f.write(tail)
            except Exception:
                pass
        time.sleep(60)
if not getattr(builtins, '_log_mirror_on', False):
    builtins._log_mirror_on = True
    threading.Thread(target=_mirror_logs, daemon=True).start()
# Hugging Face login from the Colab secret HF_TOKEN (key icon, left sidebar), if you added one; optional here
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('Hugging Face login ok')
except Exception as e:
    print('no HF_TOKEN secret, continuing without Hugging Face login:', type(e).__name__)

In [ ]:
import os; os.makedirs('/content/scripts', exist_ok=True); print('ok')

In [ ]:
%%writefile /content/scripts/droid_selfcal.py
"""Real-image self-calibration on DROID: estimate an exterior camera's pose from the robot's own body in REAL frames,
and score it against DROID's post-hoc calibrated extrinsics (KarlP/droid, cam2base_extrinsic_superset.json).

  export     droid_100 RLDS (Colab system python, TF + tfds): episodes that have calibrated extrinsics; per camera, K
             frames spread over the episode with joint angles, the calibrated cam->base pose and the intrinsics.
  segment    text-prompted robot masks on the real frames: OWLv2 box ("a robot arm") -> SAM mask (transformers).
  calibrate  render the Franka (mujoco_menagerie panda_nohand; DROID's Robotiq gripper is not modelled, so the score is
             one-sided: rendered arm boundary -> observed boundary, plus rendered pixels outside the observed mask),
             grid over viewpoints around the workspace + CEM over the 6-DoF pose using all K frames (the demo's own arm
             motion is the sweep). First checks the extrinsics convention on the calibrated poses (render IoU).
  report     position / rotation error against the calibrated extrinsics, success rates at 5 cm / 5 deg and 10 cm / 10 deg,
             and error vs the dataset's own calibration quality metric.
The calibrated extrinsics are themselves estimates (IoU-scored), so errors are measured against a reference, not truth.
"""
import argparse
import glob
import json
import os
import re

import numpy as np

HF_REPO = "KarlP/droid"
CALIB_FILES = ("cam2base_extrinsic_superset", "camera_serials", "intrinsics", "episode_id_to_path")
OUTSIDE_W = 3.0
TARGET = np.array([0.45, 0.0, 0.15])          # workspace point the exterior cameras look at, base frame (search prior)


# ------------------------------------------------------------------ geometry
def euler_xyz_to_mat(e):
    from scipy.spatial.transform import Rotation
    return Rotation.from_euler("xyz", e).as_matrix()


def mat_to_euler_xyz(R):
    from scipy.spatial.transform import Rotation
    return Rotation.from_matrix(R).as_euler("xyz")


def pose_err(C1, R1, C2, R2):
    from scipy.spatial.transform import Rotation
    return float(np.linalg.norm(C1 - C2) * 100), float(np.rad2deg(Rotation.from_matrix(R1 @ R2.T).magnitude()))


def look_at(C, target, up=np.array([0.0, 0.0, 1.0])):
    """OpenCV camera (x right, y down, z forward) at C looking at target: columns of R are camera axes in base."""
    z = target - C
    z = z / np.linalg.norm(z)
    x = np.cross(z, up)
    x = x / np.linalg.norm(x)
    y = np.cross(z, x)
    return np.stack([x, y, z], 1)


def pan_tilt(R_cv, dx, dy, f):
    """Rotate an OpenCV camera (columns of R = camera axes in base) about its own y then x axis so that the scene moves
    by (dx, dy) pixels in the image (u right, v down). Position unchanged: orientation is decoupled from the grid."""
    th, ph = -np.arctan2(dx, f), np.arctan2(dy, f)
    Ry = np.array([[np.cos(th), 0, np.sin(th)], [0, 1, 0], [-np.sin(th), 0, np.cos(th)]])
    Rx = np.array([[1, 0, 0], [0, np.cos(ph), -np.sin(ph)], [0, np.sin(ph), np.cos(ph)]])
    return R_cv @ Ry @ Rx


def best_shift(obs_masks, ren_masks, sigma=2.0):
    """The image translation (dx, dy) that best aligns the rendered masks with the observed ones, shared across the
    frames (one camera rotation moves every frame alike): peak of the summed FFT cross-correlation of blurred masks."""
    import cv2
    acc = None
    for o, r in zip(obs_masks, ren_masks):
        A = np.fft.rfft2(cv2.GaussianBlur(o.astype(np.float32), (0, 0), sigma))
        B = np.fft.rfft2(cv2.GaussianBlur(r.astype(np.float32), (0, 0), sigma))
        c = np.fft.irfft2(A * np.conj(B), s=o.shape)
        acc = c if acc is None else acc + c
    H, W = acc.shape
    iy, ix = np.unravel_index(int(np.argmax(acc)), acc.shape)
    return (ix if ix <= W // 2 else ix - W), (iy if iy <= H // 2 else iy - H)


def cv_to_mj(R_cv):
    """MuJoCo cameras look along -z with +y up: flip the OpenCV y and z axes."""
    return R_cv @ np.diag([1.0, -1.0, -1.0])


def mat_to_quat_wxyz(R):
    from scipy.spatial.transform import Rotation
    x, y, z, w = Rotation.from_matrix(R).as_quat()
    return np.array([w, x, y, z])


# ------------------------------------------------------------------ export (TF / tfds)
def rel_path_key(file_path):
    m = re.search(r"([^/]+/(?:success|failure)/\d{4}-\d{2}-\d{2}/[^/]+)/trajectory", file_path)
    return m.group(1) if m else None


def export(a):
    import tensorflow as tf
    import tensorflow_datasets as tfds
    from huggingface_hub import hf_hub_download
    tf.config.set_visible_devices([], "GPU")
    cal = {k: json.load(open(hf_hub_download(HF_REPO, f"{k}.json"))) for k in CALIB_FILES}
    # several episode ids can share one path in DROID's mapping; only unambiguous paths are used, never guessed
    by_path = {}
    for k, v in cal["episode_id_to_path"].items():
        by_path.setdefault(v.strip("/"), []).append(k)
    path_to_id = {p: ks[0] for p, ks in by_path.items() if len(ks) == 1}
    print(f"episode_id_to_path: {len(cal['episode_id_to_path'])} ids, {len(by_path)} paths, "
          f"{len(path_to_id)} unambiguous", flush=True)
    # fallback on (lab, timestamp folder) only where that pair is unique; ambiguous pairs are dropped, never guessed
    pairs = {}
    for k, v in cal["episode_id_to_path"].items():
        parts = v.strip("/").split("/")
        if len(parts) == 4:
            pairs.setdefault((parts[0], parts[3]), []).append(k)
    tail_to_id = {p: ks[0] for p, ks in pairs.items() if len(ks) == 1}
    hits = tf.io.gfile.glob(a.rlds.rstrip("/") + "/*/dataset_info.json") or tf.io.gfile.glob(a.rlds.rstrip("/") + "/dataset_info.json")
    assert hits, f"no RLDS dataset under {a.rlds}"
    ds = tfds.builder_from_directory(os.path.dirname(hits[0])).as_dataset(split="train")
    os.makedirs(a.out_dir, exist_ok=True)
    seen = matched = saved = 0
    for ep in ds:
        seen += 1
        fp = ep["episode_metadata"]["file_path"].numpy().decode()
        rel = rel_path_key(fp)
        if seen <= 3:
            print("example file_path:", fp, "->", rel, flush=True)
        eid = path_to_id.get(rel) if rel else None
        if eid is None and rel:
            parts = rel.split("/")
            eid = tail_to_id.get((parts[0], parts[3]))
        if eid is None or eid not in cal["cam2base_extrinsic_superset"] or eid not in cal["camera_serials"]:
            continue
        matched += 1
        steps = list(ep["steps"].as_numpy_iterator())
        idx = np.linspace(0, len(steps) - 1, a.frames).round().astype(int)
        for cam_key, img_key in (("ext1_cam_serial", "exterior_image_1_left"), ("ext2_cam_serial", "exterior_image_2_left")):
            serial = cal["camera_serials"][eid][cam_key]
            ext = cal["cam2base_extrinsic_superset"][eid].get(serial)
            intr = cal["intrinsics"].get(eid, {}).get(serial)
            if ext is None or intr is None:
                continue
            # the live file stores {cameraMatrix: [fx, cx, fy, cy], width, height}; the README documents a bare 4-list
            if isinstance(intr, dict):
                cm, wh = intr["cameraMatrix"], [intr["width"], intr["height"]]
            else:
                cm, wh = intr, [2 * intr[1], 2 * intr[3]]                 # principal point ~ image centre
            np.savez_compressed(
                os.path.join(a.out_dir, f"{eid.replace('+', '_')}_{serial}.npz"),
                img=np.stack([steps[i]["observation"][img_key] for i in idx]),
                q=np.stack([steps[i]["observation"]["joint_position"] for i in idx]).astype(np.float64),
                ext=np.asarray(ext, float), intr=np.asarray(cm, float),
                intr_wh=np.array(wh, float), eid=np.array(eid), serial=np.array(serial),
                quality=np.array(float(cal["cam2base_extrinsic_superset"][eid].get(f"{serial}_quality_metric", np.nan))))
            saved += 1
        if saved >= a.max_cameras:
            break
    print(f"episodes read {seen}, with calibrated extrinsics {matched}, cameras saved {saved} -> {a.out_dir}")


# ------------------------------------------------------------------ segment (transformers)
def segment(a):
    import torch
    from PIL import Image
    from transformers import Owlv2ForObjectDetection, Owlv2Processor, SamModel, SamProcessor
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    owl_p = Owlv2Processor.from_pretrained("google/owlv2-base-patch16-ensemble")
    owl = Owlv2ForObjectDetection.from_pretrained("google/owlv2-base-patch16-ensemble").to(dev).eval()
    sam_p = SamProcessor.from_pretrained("facebook/sam-vit-base")
    sam = SamModel.from_pretrained("facebook/sam-vit-base").to(dev).eval()
    queries = [["a robot arm", "a black robotic arm", "an industrial robot"]]
    for p in sorted(glob.glob(os.path.join(a.out_dir, "*.npz"))):
        d = dict(np.load(p, allow_pickle=True))
        if "mask" in d:
            continue
        masks = []
        for img in d["img"]:
            big = Image.fromarray(img).resize((img.shape[1] * 3, img.shape[0] * 3), Image.BICUBIC)
            with torch.no_grad():
                inp = owl_p(text=queries, images=big, return_tensors="pt").to(dev)
                out = owl(**inp)
                # OWLv2 pads the image to a square; box scaling by the padded side is the same in every transformers
                # version, unlike target_sizes (older releases squash the y axis on non-square frames)
                # transformers renamed the OWLv2 post-processor; both return boxes normalised to the padded square
                pp = getattr(owl_p, "post_process_object_detection", None) or owl_p.post_process_grounded_object_detection
                res = pp(out, threshold=0.05, target_sizes=None)[0]
            if len(res["scores"]) == 0:
                masks.append(np.zeros(img.shape[:2], np.uint8))
                continue
            side = float(max(big.size))
            box = (np.asarray(res["boxes"][int(torch.argmax(res["scores"]))].tolist()) * side)
            box = np.clip(box, 0, [big.size[0], big.size[1], big.size[0], big.size[1]]).tolist()
            with torch.no_grad():
                si = sam_p(big, input_boxes=[[box]], return_tensors="pt").to(dev)
                so = sam(**si)
                m = sam_p.image_processor.post_process_masks(so.pred_masks.cpu(), si["original_sizes"].cpu(),
                                                            si["reshaped_input_sizes"].cpu())[0][0]
            best = m[int(torch.argmax(so.iou_scores[0, 0]))].numpy().astype(np.uint8)
            m_full = np.asarray(Image.fromarray(best * 255).resize((img.shape[1], img.shape[0]), Image.NEAREST)) > 127
            masks.append(clean_mask(m_full) > 0)                # largest component, holes filled: no speckle edges
        d["mask"] = np.stack(masks).astype(np.uint8)
        np.savez_compressed(p, **d)
        print(f"{os.path.basename(p)}: robot pixel fraction {d['mask'].mean():.3f}", flush=True)


# ------------------------------------------------------------------ calibrate (mujoco)
class Franka:
    def __init__(self, menagerie, W=160, H=90, hand=True):
        """hand=True renders menagerie's panda.xml (Franka hand, fingers open) as a bulk proxy for DROID's Robotiq
        gripper: the observed silhouette always contains a gripper, and a hand-less render is systematically smaller,
        which biases the size term toward a farther camera (2026-09-18: estimates farther than the reference on 44 of
        53 cameras, 72% of the error along the viewing ray)."""
        import mujoco
        src = os.path.join(menagerie, "franka_emika_panda", "panda.xml" if hand else "panda_nohand.xml")
        xml = open(src).read()
        xml = xml.replace("<worldbody>", '<worldbody>\n    <camera name="cal" pos="1 0 1" xyaxes="0 -1 0 0 0 1" fovy="60"/>', 1)
        path = os.path.join(os.path.dirname(src), "_panda_cal.xml" if hand else "_panda_nohand_cal.xml")
        open(path, "w").write(xml)
        self.mj = mujoco
        self.hand = hand
        self.model = mujoco.MjModel.from_xml_path(path)
        self.data = mujoco.MjData(self.model)
        self.cid = self.model.camera("cal").id
        self.W, self.H = W, H
        self.r = mujoco.Renderer(self.model, height=H, width=W)
        self.r.enable_segmentation_rendering()

    def mask(self, q, C, R_cv, fy_px):
        m = self.model
        m.cam_pos[self.cid] = C
        m.cam_quat[self.cid] = mat_to_quat_wxyz(cv_to_mj(R_cv))
        m.cam_fovy[self.cid] = float(np.rad2deg(2 * np.arctan(0.5 * self.H / fy_px)))
        self.data.qpos[:7] = q
        if self.hand and self.model.nq >= 9:
            self.data.qpos[7:9] = 0.04                     # fingers open
        self.mj.mj_forward(m, self.data)
        self.r.update_scene(self.data, camera=self.cid)
        seg = self.r.render()
        return (seg[..., 0] >= 0).astype(np.uint8)


def clean_mask(mask, min_px=30, keep_frac=0.0):
    """Largest connected component with holes filled (uint8 0/1); tiny masks returned as is. keep_frac > 0 keeps every
    component at least that share of the largest: an arm cut by the frame border, or a gripper seen apart from the arm,
    is several components (the same lesson as the LIBERO learned masks; largest-only cost 0.15 recall on a DROID camera)."""
    import cv2
    m = (np.asarray(mask) > 0).astype(np.uint8)
    if m.sum() < min_px:
        return m
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n > 2:
        area = stats[1:, cv2.CC_STAT_AREA]
        keep = 1 + (np.where(area >= keep_frac * area.max())[0] if keep_frac > 0 else np.array([int(np.argmax(area))]))
        m = np.isin(lab, keep).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats((1 - m).astype(np.uint8), connectivity=4)
    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])))
    for k in range(1, n):
        if k not in border:
            m[lab == k] = 1
    return m


def motion_masks(imgs, sam_masks, thr=28.0, grow=25):
    """Complete the segmenter's masks with the robot's own motion: across the stored frames the room is static and
    the arm is not, so pixels that differ from the per-pixel median background belong to the arm (links, gripper,
    cables, whatever colour), restricted to the neighbourhood of the segmenter's masks so people and moved objects
    elsewhere in the frame do not enter. Union with the segmenter mask (which covers the static base), then cleaned.
    Self-measurement: the robot knows it moved. Returns uint8 0/1 masks at the frame resolution."""
    import cv2
    imgs = np.asarray(imgs).astype(np.float32)
    sam = np.asarray(sam_masks) > 0
    # background = per-pixel median over the frames in which the segmenter saw no robot there (a plain median turns
    # into the arm wherever the arm parks for half the frames, and then the arm's absence reads as motion)
    free = np.where(sam[..., None], np.nan, imgs)
    with np.errstate(all="ignore"):
        bg = np.nanmedian(free, axis=0)
    bg = np.where(np.isnan(bg), np.median(imgs, axis=0), bg)
    region = cv2.dilate(np.any(np.asarray(sam_masks) > 0, axis=0).astype(np.uint8), np.ones((grow, grow), np.uint8))
    out = []
    for im, sm in zip(imgs, sam_masks):
        diff = np.abs(im - bg).max(-1)
        diff = cv2.GaussianBlur(diff, (5, 5), 0)
        mv = (diff > thr).astype(np.uint8) & region
        out.append(clean_mask((mv | (np.asarray(sm) > 0).astype(np.uint8))))
    return out


def soft_iou(a_, b_, sigma=2.0):
    """IoU of Gaussian-blurred masks: indifferent to boundary detail, so a ragged real mask and a clean render can
    still agree; the coarse search and the CEM both optimise 1 - soft_iou."""
    import cv2
    A = cv2.GaussianBlur(a_.astype(np.float32), (0, 0), sigma)
    B = cv2.GaussianBlur(b_.astype(np.float32), (0, 0), sigma)
    u = np.maximum(A, B).sum()
    return float(np.minimum(A, B).sum() / u) if u > 0 else 0.0


def one_sided(obs_mask, ren_mask):
    import cv2
    if ren_mask.sum() < 5:
        return 1e3
    edge = lambda m: m & (1 - cv2.erode(m, np.ones((3, 3), np.uint8)))
    eo, er = edge(obs_mask), edge(ren_mask)
    if eo.sum() == 0:
        return 1e3
    dt = cv2.distanceTransform((1 - eo).astype(np.uint8), cv2.DIST_L2, 3)
    outside = float((ren_mask & (1 - cv2.dilate(obs_mask, np.ones((5, 5), np.uint8)))).sum() / ren_mask.sum())
    # size term, as in the simulator calibration (calibrate_v2 area_w): without it the edge chamfer is minimised by a
    # render that explains part of the boundary and never sticks out, i.e. the estimate drifts to a smaller, wrong
    # silhouette (seen on DROID 2026-09-17: estimate score below the reference with lower IoU)
    size = abs(float(np.log((ren_mask.sum() + 1.0) / (obs_mask.sum() + 1.0))))
    # OUTSIDE_W was 20: with an incomplete real-image mask the TRUE pose renders the missing part of the arm and is
    # punished for it (DROID v2: reference scores 130 to 311 at IoU 0.46), so this is one clue among three, not a veto
    return float(dt[er > 0].mean()) + OUTSIDE_W * outside + 4.0 * size


def iou(a_, b_):
    u = (a_ | b_).sum()
    return float((a_ & b_).sum() / u) if u else 0.0


def calibrate(a):
    import cv2
    rob = Franka(a.menagerie, hand=bool(a.hand))
    out_path = os.path.join(a.out_dir, "selfcal_results.jsonl")
    done = set()
    if os.path.exists(out_path):
        done = {json.loads(l)["file"] for l in open(out_path) if l.strip()}
    files = [p for p in sorted(glob.glob(os.path.join(a.out_dir, "*.npz"))) if os.path.basename(p) not in done]
    valid = [float(np.load(p, allow_pickle=True)["intr"][2]) for p in sorted(glob.glob(os.path.join(a.out_dir, "*.npz")))]
    valid = [v for v in valid if np.isfinite(v) and v > 0]
    median_fy = float(np.median(valid)) if valid else 530.0
    print(f"intrinsics: {len(valid)} cameras with data, median fy {median_fy:.1f} px at 1280x720 (used where DROID has none)", flush=True)
    rng = np.random.default_rng(0)
    convention = a.convention
    with open(out_path, "a") as f:
        for p in files:
            d = np.load(p, allow_pickle=True)
            if "mask" not in d:
                continue
            full_masks = motion_masks(d["img"], d["mask"]) if a.motion else [clean_mask(m, keep_frac=a.keep_frac) for m in d["mask"]]
            obs = [cv2.resize(m, (rob.W, rob.H), interpolation=cv2.INTER_NEAREST).astype(np.uint8) for m in full_masks]
            sam_only = [cv2.resize(clean_mask(m), (rob.W, rob.H), interpolation=cv2.INTER_NEAREST) for m in d["mask"]]
            keep = [i for i, m in enumerate(obs) if m.mean() > 0.01]
            if len(keep) < 2:
                continue
            fx, cx, fy, cy = d["intr"]
            imputed = False
            if not (np.isfinite(fx) and fx > 0):           # DROID ships all-zero intrinsics for some cameras
                fx = fy = median_fy; imputed = True        # same camera model everywhere: the median of the valid ones
            elif not np.isfinite(fy) or fy <= 0:
                fy = fx
            wh_y = float(d["intr_wh"][1])
            if not np.isfinite(wh_y) or wh_y <= 0:          # no intrinsics record: DROID calibrates at 1280 x 720
                wh_y = 720.0
            fy_px = fy * rob.H / wh_y
            T = np.eye(4)
            T[:3, :3], T[:3, 3] = euler_xyz_to_mat(d["ext"][3:6]), d["ext"][:3]
            variants = {"cam2base": T, "base2cam": np.linalg.inv(T)}
            gt_iou = {k: float(np.mean([iou(rob.mask(d["q"][i], V[:3, 3], V[:3, :3], fy_px), obs[i]) for i in keep]))
                      for k, V in variants.items()}
            if convention == "auto":
                convention = max(gt_iou, key=gt_iou.get)
                print(f"extrinsics convention check on the first camera: render IoU {gt_iou} -> using {convention}", flush=True)
            Tg = variants[convention]
            Cg, Rg = Tg[:3, 3], Tg[:3, :3]
            chamfer = lambda C, R: float(np.mean([one_sided(obs[i], rob.mask(d["q"][i], C, R, fy_px)) for i in keep]))
            score = lambda C, R: 1.0 - float(np.mean([soft_iou(obs[i], rob.mask(d["q"][i], C, R, fy_px)) for i in keep]))
            # mask completeness: share of the reference render that the mask covers (1 = nothing of the arm is missed)
            recall_sam = float(np.mean([(rob.mask(d["q"][i], Cg, Rg, fy_px) & sam_only[i]).sum() / max(rob.mask(d["q"][i], Cg, Rg, fy_px).sum(), 1) for i in keep]))
            prec_full = float(np.mean([(rob.mask(d["q"][i], Cg, Rg, fy_px) & obs[i]).sum() / max(obs[i].sum(), 1) for i in keep]))
            recall_full = float(np.mean([(rob.mask(d["q"][i], Cg, Rg, fy_px) & obs[i]).sum() / max(rob.mask(d["q"][i], Cg, Rg, fy_px).sum(), 1) for i in keep]))

            def refine(Cb, Rb, iters=10):
                mu, sd = np.zeros(6), np.array([0.10, 0.10, 0.10, 0.35, 0.35, 0.35])
                bestl = (score(Cb, Rb), Cb, Rb)
                for _ in range(iters):
                    X = mu + sd * rng.standard_normal((32, 6))
                    X[0] = mu
                    vals = []
                    for x in X:
                        C, R = Cb + x[:3], euler_xyz_to_mat(x[3:]) @ Rb
                        v = score(C, R)
                        vals.append(v)
                        if v < bestl[0]:
                            bestl = (v, C, R)
                    el_ = X[np.argsort(vals)[:6]]
                    mu, sd = el_.mean(0), el_.std(0) + 1e-3
                return bestl
            cands = []
            # DROID's exterior cameras sit 0.44 to 0.86 m from the workspace point (median 0.63) at 2 to 45 deg
            # elevation (53 reference poses, 2026-09-18); the old grid started at 0.7 m and 15 deg, outside 36 of 53
            for rad in (0.4, 0.55, 0.7, 0.9, 1.15):
                for az in range(0, 360, 20):
                    for el in (0, 10, 20, 30, 45):
                        e, zr = np.deg2rad(el), np.deg2rad(az)
                        C = TARGET + rad * np.array([np.cos(e) * np.cos(zr), np.cos(e) * np.sin(zr), np.sin(e)])
                        R = look_at(C, TARGET)
                        # real mounts point up to 50+ deg away from the look-at prior, so the robot sits off-centre and
                        # a centred render scores ~0 even at the right POSITION (v5: from-reference refinement held
                        # within 5 cm while the search landed 50 cm / 85 deg away). Align the render to the observed
                        # masks by an image shift, turn that shift into a pan/tilt, score the re-aimed camera.
                        rens = [rob.mask(d["q"][i], C, R, fy_px) for i in keep]
                        if sum(int(m.sum()) for m in rens) > 0:
                            dx, dy = best_shift([obs[i] for i in keep], rens)
                            R = pan_tilt(R, dx, dy, fy_px)
                        cands.append((score(C, R), C, R))
            cands.sort(key=lambda x: x[0])
            best = cands[0]
            seeds = []                      # distinct basins: skip a candidate within 15 cm and 20 deg of a chosen seed
            for cnd in cands:
                if all(pose_err(cnd[1], cnd[2], s_[1], s_[2])[0] > 15 or pose_err(cnd[1], cnd[2], s_[1], s_[2])[1] > 20
                       for s_ in seeds):
                    seeds.append(cnd)
                if len(seeds) == 5:
                    break
            for s0, Cb, Rb in seeds:
                r_ = refine(Cb, Rb, iters=8)
                if r_[0] < best[0]:
                    best = r_
            _, Ce, Re = best
            err = pose_err(Ce, Re, Cg, Rg)
            # diagnostic: refine FROM the reference. If it stays put and scores no worse than the estimate, the
            # estimate sits in another basin (search); if it drifts away, the objective itself prefers a wrong pose (mask)
            ref_ref = refine(Cg, Rg, iters=6)
            ref_err = pose_err(ref_ref[1], ref_ref[2], Cg, Rg)
            z_view = Re[:, 2]
            dvec = Ce - Cg
            along = float(abs(dvec @ z_view)) * 100
            lateral = float(np.linalg.norm(dvec - (dvec @ z_view) * z_view)) * 100
            row = dict(file=os.path.basename(p), eid=str(d["eid"]), serial=str(d["serial"]), convention=convention,
                       intr_imputed=imputed,
                       quality=float(d["quality"]), frames=len(keep), gt_render_iou=gt_iou[convention],
                       est_render_iou=float(np.mean([iou(rob.mask(d["q"][i], Ce, Re, fy_px), obs[i]) for i in keep])),
                       score_gt=score(Cg, Rg), score_est=best[0], chamfer_gt=chamfer(Cg, Rg), chamfer_est=chamfer(Ce, Re),
                       search="v6", pos_err_cm=err[0], rot_err_deg=err[1], err_along_cm=along, err_lateral_cm=lateral,
                       mask_recall_sam=recall_sam, mask_recall_full=recall_full, mask_precision_full=prec_full,
                       ref_refined_err_cm=ref_err[0], ref_refined_err_deg=ref_err[1], score_ref_refined=ref_ref[0],
                       est_pos=Ce.tolist(), est_euler=mat_to_euler_xyz(Re).tolist(), gt_pos=Cg.tolist())
            f.write(json.dumps(row) + "\n")
            f.flush()
            print(f"{row['file']}: error {err[0]:.1f} cm / {err[1]:.1f} deg (along {along:.0f}, lateral {lateral:.0f}) | "
                  f"IoU gt {row['gt_render_iou']:.2f} est {row['est_render_iou']:.2f} | mask recall sam {recall_sam:.2f} "
                  f"full {recall_full:.2f} precision {prec_full:.2f} | from-reference refine stays within {ref_err[0]:.1f} cm", flush=True)


def report(a):
    rows = [json.loads(l) for l in open(os.path.join(a.out_dir, "selfcal_results.jsonl")) if l.strip()]
    pe, re_ = np.array([r["pos_err_cm"] for r in rows]), np.array([r["rot_err_deg"] for r in rows])
    print(f"cameras {len(rows)} | median error {np.median(pe):.1f} cm / {np.median(re_):.1f} deg")
    for c, g in ((5, 5), (10, 10), (20, 15)):
        print(f"  within {c} cm and {g} deg: {np.mean((pe <= c) & (re_ <= g)):.2f}")
    gi = np.array([r["gt_render_iou"] for r in rows])
    print(f"render IoU with the reference extrinsics (convention sanity): median {np.median(gi):.2f}")
    good = gi >= np.median(gi)
    print(f"  on cameras where the reference itself renders well (IoU >= median): median error "
          f"{np.median(pe[good]):.1f} cm / {np.median(re_[good]):.1f} deg")
    worse = np.mean([r["score_est"] <= r["score_gt"] for r in rows])
    print(f"estimate explains the observed silhouette at least as well as the reference in {worse:.0%} of cameras")
    if "err_along_cm" in rows[0]:
        al = np.array([r["err_along_cm"] for r in rows]); la = np.array([r["err_lateral_cm"] for r in rows])
        rr = np.array([r["ref_refined_err_cm"] for r in rows])
        rs = np.array([r["mask_recall_sam"] for r in rows]); rf = np.array([r["mask_recall_full"] for r in rows])
        print(f"error split: along the viewing ray {np.median(al):.0f} cm, lateral {np.median(la):.0f} cm (medians)")
        print(f"mask completeness (share of the reference render covered): segmenter {np.median(rs):.2f} -> with motion {np.median(rf):.2f}")
        print(f"refining from the reference stays within 5 cm on {np.mean(rr <= 5):.0%} of cameras (objective consistent with "
              f"the reference there); drifts beyond 20 cm on {np.mean(rr > 20):.0%} (objective prefers another pose: mask or reference)")
    # DROID publishes a per-camera calibration quality metric; values in the tens of thousands mark a reference that
    # DROID itself flags as unreliable. Report the trustworthy references separately.
    q = np.array([r["quality"] for r in rows])
    imp = np.array([bool(r.get("intr_imputed", False)) for r in rows])
    ok = np.isfinite(q) & (q < 20.0) & ~imp
    if imp.any():
        print(f"intrinsics imputed on {int(imp.sum())} cameras (DROID has none): median error {np.median(pe[imp]):.1f} cm, "
              f"reported separately")
    if ok.any():
        print(f"reference flagged reliable by DROID (quality < 20): {int(ok.sum())} cameras | median error "
              f"{np.median(pe[ok]):.1f} cm / {np.median(re_[ok]):.1f} deg | within 20 cm and 15 deg: "
              f"{np.mean((pe[ok] <= 20) & (re_[ok] <= 15)):.2f} | reference IoU median {np.median(gi[ok]):.2f}")
        bad = ~ok
        if bad.any():
            print(f"reference flagged unreliable ({int(bad.sum())} cameras): median error {np.median(pe[bad]):.1f} cm; "
                  f"errors against these are not evidence either way")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["export", "segment", "calibrate", "report"])
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--rlds", default="gs://gresearch/robotics/droid_100")
    ap.add_argument("--frames", type=int, default=8)
    ap.add_argument("--max_cameras", type=int, default=60)
    ap.add_argument("--motion", type=int, default=1, help="complete the segmenter masks with the robot's own motion (v5)")
    ap.add_argument("--keep_frac", type=float, default=0.0,
                    help="with --motion 0: keep every mask component at least this share of the largest (0 = largest only, v6)")
    ap.add_argument("--hand", type=int, default=1, help="render the Franka hand as a gripper proxy (v5)")
    ap.add_argument("--menagerie", default="/content/mujoco_menagerie")
    ap.add_argument("--convention", default="auto", choices=["auto", "cam2base", "base2cam"])
    a = ap.parse_args()
    dict(export=export, segment=segment, calibrate=calibrate, report=report)[a.mode](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/droid_vlm_seg.py
"""Robot masks for the DROID self-calibration chosen by a vision-language model (set-of-marks), replacing the zero-shot
OWLv2 box -> SAM masks whose background pixels capped the real-image result (mask precision 0.46 to 0.57).

A VLM is poor at drawing a mask and good at saying what a region is, so the work is split:
  propose   SAM's automatic masks cut every frame into a COMPLETE partition of numbered regions (GPU helps, CPU works)
  select    the VLM sees the frame and the numbered frame and returns the region numbers that are robot (API only)
  refine    every region the VLM called part robot, part background is cropped, zoomed and cut again (optional)
  select --round 2   the same question on the zoomed crops
  assemble  the chosen regions become the mask; the camera files are COPIED to --out_dir with the new `mask`
            (the old one is kept as `mask_owl`), so `droid_selfcal.py calibrate --out_dir <out_dir>` runs unchanged
  eval      old and new masks against the robot rendered at DROID's reference pose (needs mujoco + menagerie)

    python droid_vlm_seg.py propose  --in_dir <droid_selfcal> --work_dir <work>
    python droid_vlm_seg.py select   --in_dir <droid_selfcal> --work_dir <work>          # key in ANTHROPIC_API_KEY
    python droid_vlm_seg.py refine   --in_dir <droid_selfcal> --work_dir <work>          # optional, then select --round 2
    python droid_vlm_seg.py assemble --in_dir <droid_selfcal> --work_dir <work> --out_dir <droid_selfcal_vlm>
    python droid_vlm_seg.py eval     --in_dir <droid_selfcal> --out_dir <droid_selfcal_vlm> --menagerie <path>

Every stage is resumable. The key is read from the environment only and is never written anywhere.
"""
import argparse
import base64
import glob
import json
import os
import sys
import threading
import time

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

UP = 3                      # SAM runs on the frame upscaled 3x (the stored frames are 320 x 180)
SEND_W = 1280               # width of the two images the VLM sees
MIN_FRAC = 0.0015           # regions smaller than this share of the frame are not numbered
MAX_REGIONS = 80
PRICE = {"claude-opus-5": (5.0, 25.0), "claude-sonnet-5": (2.0, 10.0), "claude-haiku-4-5": (1.0, 5.0),
         "claude-fable-5-1": (10.0, 50.0)}          # USD per million tokens (input, output)
PALETTE = [(255, 64, 64), (64, 200, 64), (64, 128, 255), (255, 200, 0), (255, 0, 255), (0, 220, 220), (255, 128, 0),
           (160, 96, 255), (0, 160, 96), (255, 128, 192)]

SYSTEM = (
    "You label image regions for a robot self-calibration pipeline. Each camera frame shows a lab or home scene with a "
    "Franka Emika Panda arm (white links, black joints and wrist) carrying a Robotiq two-finger gripper (black). The arm "
    "may be cut by the image border, may be small, or may be absent. You receive the plain frame and the same frame cut "
    "into numbered regions (tinted, outlined, number inside the region). Your answer is compared against a render of "
    "the arm and gripper ALONE, so a region counts as robot only if it is part of the arm's links, joints, wrist or the "
    "gripper and its fingers. NOT robot: cables and hoses (also those hanging from the arm), the table, stand, pedestal "
    "or control box the arm is bolted to, cameras and their mounts or tripods (also a camera on the wrist when it is its "
    "own region), any object held or touched, shadows, reflections, other equipment, and people.")

QUESTION = (
    "Image 1 is the camera frame. Image 2 is the same frame with numbered regions. Return:\n"
    "robot_visible: whether any part of the arm or gripper is in view.\n"
    "robot_ids: every region number of which at least about 80 percent is arm or gripper.\n"
    "mixed_ids: region numbers that contain a substantial part of the arm or gripper AND a substantial part of "
    "something else (for example the arm merged with the wall behind it).\n"
    "Check every number near the arm, including small regions on the gripper and regions where the arm leaves the frame. "
    "Use only numbers that appear in image 2.")

SCHEMA = {
    "type": "object",
    "properties": {
        "robot_visible": {"type": "boolean"},
        "robot_ids": {"type": "array", "items": {"type": "integer"}},
        "mixed_ids": {"type": "array", "items": {"type": "integer"}},
    },
    "required": ["robot_visible", "robot_ids", "mixed_ids"],
    "additionalProperties": False,
}


# ------------------------------------------------------------------ regions
def regions_from_masks(masks, shape, min_frac=MIN_FRAC, max_regions=MAX_REGIONS):
    """A complete partition of the frame from overlapping SAM masks: larger masks are painted first so every pixel ends
    in the SMALLEST mask that contains it; what no mask covers is split into connected components. Regions under
    min_frac of the frame, and those beyond the max_regions largest, get label 0 (not selectable).
    Returns (labels int16, share of the frame left unlabelled)."""
    import cv2
    H, W = shape
    lab = np.zeros((H, W), np.int32)
    order = sorted(range(len(masks)), key=lambda i: -int(np.asarray(masks[i]).sum()))
    for n, i in enumerate(order, 1):
        lab[np.asarray(masks[i]) > 0] = n
    n_cc, cc = cv2.connectedComponents((lab == 0).astype(np.uint8), connectivity=4)
    top = int(lab.max())
    for k in range(1, n_cc):
        lab[cc == k] = top + k
    area = np.bincount(lab.ravel())
    keep = [i for i in np.argsort(-area) if i > 0 and area[i] >= min_frac * H * W][:max_regions]
    out = np.zeros((H, W), np.int16)
    for new, old in enumerate(keep, 1):
        out[lab == old] = new
    return out, float((out == 0).mean())


def draw_marks(img, lab, font=0.45):
    """The frame with every region tinted, outlined and numbered at its deepest interior point."""
    import cv2
    out = img.astype(np.float32)
    ids = [int(i) for i in np.unique(lab) if i > 0]
    for i in ids:
        m = lab == i
        out[m] = 0.84 * out[m] + 0.16 * np.array(PALETTE[i % len(PALETTE)], np.float32)   # light: the object stays readable
    out = np.clip(out, 0, 255).astype(np.uint8)
    for i in ids:
        cs, _ = cv2.findContours((lab == i).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(out, cs, -1, PALETTE[i % len(PALETTE)], 1)
    for i in ids:
        m = np.pad((lab == i).astype(np.uint8), 1)                       # the pad makes the image border count as an edge
        dt = cv2.distanceTransform(m, cv2.DIST_L2, 3)[1:-1, 1:-1]
        y, x = np.unravel_index(int(np.argmax(dt)), dt.shape)
        text = str(i)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font, 1)
        x0 = int(np.clip(x - tw // 2, 1, out.shape[1] - tw - 2))
        y0 = int(np.clip(y + th // 2, th + 2, out.shape[0] - 2))
        cv2.rectangle(out, (x0 - 1, y0 - th - 2), (x0 + tw + 1, y0 + 2), (0, 0, 0), -1)
        cv2.putText(out, text, (x0, y0), cv2.FONT_HERSHEY_SIMPLEX, font, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def views(img, lab, inside=None):
    """(plain, marked) RGB images at the size the VLM sees (long side SEND_W); lab is the label map at any resolution.
    inside: optional mask of the area in question; the rest of the marked image is dimmed."""
    import cv2
    k = SEND_W / max(img.shape[:2])
    w, h = int(round(img.shape[1] * k)), int(round(img.shape[0] * k))
    plain = cv2.resize(img, (w, h), interpolation=cv2.INTER_CUBIC)
    marked = draw_marks(plain, cv2.resize(lab, (w, h), interpolation=cv2.INTER_NEAREST), font=0.45 if inside is None else 0.6)
    if inside is not None:
        out = cv2.resize(inside.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) == 0
        marked[out] = (0.35 * marked[out]).astype(np.uint8)
    return plain, marked


def frames_of(a, n):
    return list(range(n)) if a.frames == "all" else [int(x) for x in a.frames.split(",") if int(x) < n]


def camera_files(a):
    ps = sorted(glob.glob(os.path.join(a.in_dir, "*.npz")))
    return ps[:a.limit] if a.limit else ps


def props_path(a, p, kind="props"):
    return os.path.join(a.work_dir, os.path.basename(p).replace(".npz", f".{kind}.npz"))


def upscaled(img):
    from PIL import Image
    return np.asarray(Image.fromarray(img).resize((img.shape[1] * UP, img.shape[0] * UP), Image.BICUBIC))


def sam_generator(a):
    import torch
    from transformers import pipeline
    gpu = torch.cuda.is_available()
    name = a.sam or ("facebook/sam-vit-huge" if gpu else "facebook/sam-vit-base")
    print("SAM:", name, "on", "cuda" if gpu else "cpu", flush=True)
    return pipeline("mask-generation", model=name, device=0 if gpu else -1)


# ------------------------------------------------------------------ propose (transformers)
def propose(a):
    import cv2
    import torch
    from PIL import Image
    os.makedirs(a.work_dir, exist_ok=True)
    gen = sam_generator(a)
    for p in camera_files(a):
        out = props_path(a, p)
        if os.path.exists(out):
            continue
        d = np.load(p, allow_pickle=True)
        labs, lost = [], []
        for t, img in enumerate(d["img"]):
            if t not in frames_of(a, len(d["img"])):
                labs.append(np.zeros((img.shape[0] * UP, img.shape[1] * UP), np.int16))
                continue
            big = Image.fromarray(img).resize((img.shape[1] * UP, img.shape[0] * UP), Image.BICUBIC)
            with torch.no_grad():
                res = gen(big, points_per_batch=a.points_per_batch, pred_iou_thresh=0.86, stability_score_thresh=0.90)
            lab, miss = regions_from_masks([np.asarray(m) for m in res["masks"]], (big.size[1], big.size[0]))
            labs.append(lab)
            lost.append(miss)
        np.savez_compressed(out, labels=np.stack(labs))
        print(f"{os.path.basename(p)}: {int(np.mean([l.max() for l in labs]))} regions per frame, "
              f"{100 * float(np.mean(lost)):.1f}% of the frame in regions too small to number", flush=True)
        if a.dump_marked:
            t = frames_of(a, len(d["img"]))[0]
            _, marked = views(d["img"][t], labs[t])
            cv2.imwrite(out.replace(".props.npz", f".f{t}.marked.jpg"), cv2.cvtColor(marked, cv2.COLOR_RGB2BGR))


# ------------------------------------------------------------------ refine (transformers)
def refine(a):
    """Zoom into every region the VLM called mixed (the arm merged with the wall behind it, say): crop it, upscale the
    crop, cut it again with SAM. The crops are asked about with `select --round 2`."""
    import cv2
    import torch
    from PIL import Image
    _, sel = read_selection(a)
    gen = None
    for p in camera_files(a):
        out = props_path(a, p, "refine")
        todo = [(t, rid) for (f, t, reg), r in sel.items() if f == os.path.basename(p) and reg == 0 and r["robot_visible"]
                for rid in r["mixed_ids"][:a.max_mixed]]
        if os.path.exists(out) or not todo or not os.path.exists(props_path(a, p)):
            continue
        gen = gen or sam_generator(a)
        d = np.load(p, allow_pickle=True)
        labs = np.load(props_path(a, p))["labels"]
        store = {}
        for t, rid in sorted(todo):
            parent = labs[t] == rid
            near = np.isin(labs[t], sel[(os.path.basename(p), t, 0)]["robot_ids"])
            if near.any():                  # a mixed region can be the whole wall: only its part next to the robot regions
                ys, xs = np.where(near)     # found so far is worth a zoom (their box, grown by half its size)
                gx, gy = (xs.max() - xs.min()) // 2 + 30, (ys.max() - ys.min()) // 2 + 30
                box = np.zeros_like(parent)
                box[max(ys.min() - gy, 0):ys.max() + gy, max(xs.min() - gx, 0):xs.max() + gx] = True
                parent = parent & box
            ys, xs = np.where(parent)
            if len(ys) == 0:
                continue
            padx, pady = int(0.08 * (xs.max() - xs.min() + 1)) + 4, int(0.08 * (ys.max() - ys.min() + 1)) + 4
            x0, x1 = max(xs.min() - padx, 0), min(xs.max() + padx + 1, parent.shape[1])
            y0, y1 = max(ys.min() - pady, 0), min(ys.max() + pady + 1, parent.shape[0])
            crop = upscaled(d["img"][t])[y0:y1, x0:x1]
            k = min(4.0, 1024.0 / max(crop.shape[:2]))
            big = Image.fromarray(crop).resize((max(int(crop.shape[1] * k), 8), max(int(crop.shape[0] * k), 8)), Image.BICUBIC)
            with torch.no_grad():
                res = gen(big, points_per_batch=a.points_per_batch, pred_iou_thresh=0.86, stability_score_thresh=0.90)
            lab, _ = regions_from_masks([np.asarray(m) for m in res["masks"]], (big.size[1], big.size[0]), min_frac=0.004, max_regions=40)
            inside = cv2.resize(parent[y0:y1, x0:x1].astype(np.uint8), big.size, interpolation=cv2.INTER_NEAREST) > 0
            lab[~inside] = 0                                  # only the parent region is in question
            store[f"lab_{t}_{rid}"] = lab
            store[f"box_{t}_{rid}"] = np.array([x0, y0, x1, y1])
        np.savez_compressed(out, **store)
        print(f"{os.path.basename(p)}: {len(store) // 2} mixed regions cut again", flush=True)


# ------------------------------------------------------------------ select (Anthropic API)
def jpeg_block(img):
    import cv2
    ok, buf = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 92])
    assert ok
    return {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg",
                                        "data": base64.standard_b64encode(buf.tobytes()).decode("utf-8")}}


ZOOM_NOTE = ("\n\nThis pair is a ZOOMED CROP of one region of a larger frame that holds part of the arm or gripper merged "
             "with something else. Only the numbered, undimmed area is in question; the dimmed surround is context.")


def ask(client, a, img, lab, inside=None):
    """One frame (or one zoomed crop) -> dict(robot_visible, robot_ids, mixed_ids, tokens) or dict(error=...)."""
    plain, marked = views(img, lab, inside)
    cfg = {"format": {"type": "json_schema", "schema": SCHEMA}}
    if "haiku" not in a.model:                              # Haiku 4.5 rejects the effort field
        cfg["effort"] = a.effort
    resp = client.messages.create(
        model=a.model, max_tokens=8000, system=SYSTEM, output_config=cfg,
        messages=[{"role": "user", "content": [{"type": "text", "text": "Image 1 (plain frame):"}, jpeg_block(plain),
                                               {"type": "text", "text": "Image 2 (numbered regions):"}, jpeg_block(marked),
                                               {"type": "text", "text": QUESTION + (ZOOM_NOTE if inside is not None else "")}]}])
    usage = dict(input=resp.usage.input_tokens, output=resp.usage.output_tokens)
    if resp.stop_reason != "end_turn":                      # refusal or max_tokens: no usable answer for this frame
        return dict(error=str(resp.stop_reason), tokens=usage)
    ans = json.loads(next(b.text for b in resp.content if b.type == "text"))
    valid = {int(i) for i in np.unique(lab) if i > 0}
    return dict(robot_visible=bool(ans["robot_visible"]),
                robot_ids=sorted({int(i) for i in ans["robot_ids"]} & valid),
                mixed_ids=sorted(({int(i) for i in ans["mixed_ids"]} & valid) - {int(i) for i in ans["robot_ids"]}),
                unknown_ids=sorted(({int(i) for i in ans["robot_ids"]} | {int(i) for i in ans["mixed_ids"]}) - valid),
                tokens=usage)


def read_selection(a):
    path = os.path.join(a.work_dir, "vlm_select.jsonl")
    rows = [json.loads(l) for l in open(path) if l.strip()] if os.path.exists(path) else []
    return path, {(r["file"], r["frame"], r.get("region", 0)): r for r in rows if "error" not in r}      # the last good row wins


def select(a):
    import anthropic
    from concurrent.futures import ThreadPoolExecutor
    if not (os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN")):
        sys.exit("no API key: set ANTHROPIC_API_KEY in the environment (Colab: a secret of that name)")
    client = anthropic.Anthropic(max_retries=6, timeout=180.0)
    path, done = read_selection(a)
    jobs = []
    for p in camera_files(a):
        if a.round == 2 and os.path.exists(props_path(a, p, "refine")):
            keys = [k for k in np.load(props_path(a, p, "refine")).files if k.startswith("lab_")]
            units = [(int(k.split("_")[1]), int(k.split("_")[2])) for k in keys]
            jobs += [(p, t, rid) for t, rid in sorted(units) if (os.path.basename(p), t, rid) not in done]
        elif a.round == 1 and os.path.exists(props_path(a, p)):
            n = len(np.load(p, allow_pickle=True)["img"])
            jobs += [(p, t, 0) for t in frames_of(a, n) if (os.path.basename(p), t, 0) not in done]
    print(f"{len(done)} frames already answered, {len(jobs)} to ask ({a.model}, effort {a.effort})", flush=True)
    lock, spent, halt = threading.Lock(), dict(input=0, output=0, n=0), threading.Event()

    def work(job):
        p, t, rid = job
        if halt.is_set():                                   # the key stopped working: do not burn through the queue
            return None
        d = np.load(p, allow_pickle=True)
        if rid:
            z = np.load(props_path(a, p, "refine"))
            lab, (x0, y0, x1, y1) = z[f"lab_{t}_{rid}"], z[f"box_{t}_{rid}"]
            img, inside = upscaled(d["img"][t])[y0:y1, x0:x1], lab > 0
        else:
            lab, img, inside = np.load(props_path(a, p))["labels"][t], d["img"][t], None
        try:
            r = ask(client, a, img, lab, inside)
        except anthropic.AuthenticationError as e:
            r = dict(error=f"authentication: {e}")
        except anthropic.RateLimitError as e:
            r = dict(error=f"rate limit after retries: {e}")
        except anthropic.APIStatusError as e:
            r = dict(error=f"status {e.status_code}: {e}")
        except anthropic.APIConnectionError as e:
            r = dict(error=f"connection: {e}")
        if any(w in str(r.get("error", "")) for w in ("usage limits", "authentication", "credit balance", "status 401", "status 403")):
            if not halt.is_set():
                print("STOPPING: the API key is refused (" + r["error"][:160] + "). Fix the key or its spend limit, then re-run this "
                      "cell: answered frames are kept, only the rest is asked.", flush=True)
            halt.set()
            return None
        r.update(file=os.path.basename(p), frame=t, region=rid, model=a.model)
        with lock:
            for attempt in range(6):                        # the Drive mount drops out for a moment now and then
                try:
                    os.makedirs(os.path.dirname(path), exist_ok=True)
                    with open(path, "a") as f:
                        f.write(json.dumps(r) + "\n")
                    break
                except OSError:
                    if attempt == 5:
                        raise
                    time.sleep(5)
            for k in ("input", "output"):
                spent[k] += r.get("tokens", {}).get(k, 0)
            spent["n"] += 1
            tail = r.get("error") or f"robot {r['robot_ids']} mixed {r['mixed_ids']}" + ("" if r["robot_visible"] else " (not visible)")
            print(f"[{spent['n']}/{len(jobs)}] {r['file']} f{t}" + (f" region {rid}" if rid else "") + f": {tail}", flush=True)
        return r

    with ThreadPoolExecutor(max_workers=a.workers) as ex:
        rows = [r for r in ex.map(work, jobs) if r is not None]
    pin, pout = PRICE.get(a.model, (0.0, 0.0))
    bad = sum("error" in r for r in rows)
    print(f"asked {len(rows)} frames, {bad} without an answer (re-run to retry them) | tokens in {spent['input']} out "
          f"{spent['output']} | about ${(spent['input'] * pin + spent['output'] * pout) / 1e6:.2f} at list price")


# ------------------------------------------------------------------ assemble
def motion_px(imgs, masks, thr=28.0):
    """Pixels that differ from the static background (per-pixel median over the frames in which the mask saw no robot
    there): the robot's own motion. uint8 0/1 per frame, at the stored frame size."""
    import cv2
    import warnings
    imgs = np.asarray(imgs, np.float32)
    free = np.where((np.asarray(masks) > 0)[..., None], np.nan, imgs)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")                        # all-NaN columns are expected and handled on the next line
        bg = np.nanmedian(free, axis=0)
    bg = np.where(np.isnan(bg), np.median(imgs, axis=0), bg)
    return [(cv2.GaussianBlur(np.abs(im - bg).max(-1), (5, 5), 0) > thr).astype(np.uint8) for im in imgs]


def mask_from_ids(lab, ids, shape, extra=()):
    """Chosen regions -> uint8 mask at the stored frame size; a small close bridges the unnumbered slivers between them.
    extra: (box, sub_mask) pairs from the zoomed crops of mixed regions, pasted in at the label-map resolution."""
    import cv2
    m = np.isin(lab, list(ids)).astype(np.uint8)
    for (x0, y0, x1, y1), sm in extra:
        m[y0:y1, x0:x1] |= cv2.resize(sm.astype(np.uint8), (int(x1 - x0), int(y1 - y0)), interpolation=cv2.INTER_NEAREST)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((2 * UP + 1, 2 * UP + 1), np.uint8))
    return (cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_AREA) > 0).astype(np.uint8) if m.any() else np.zeros(shape, np.uint8)


def assemble(a):
    os.makedirs(a.out_dir, exist_ok=True)
    assert os.path.abspath(a.out_dir) != os.path.abspath(a.in_dir), "--out_dir must differ from --in_dir: the old masks are kept"
    _, sel = read_selection(a)
    for p in camera_files(a):
        if not os.path.exists(props_path(a, p)):
            continue
        d = dict(np.load(p, allow_pickle=True))
        labs = np.load(props_path(a, p))["labels"]
        z = np.load(props_path(a, p, "refine")) if os.path.exists(props_path(a, p, "refine")) else None
        masks, used, zoomed = [], 0, 0
        for t, img in enumerate(d["img"]):
            name = os.path.basename(p)
            r = sel.get((name, t, 0))
            ids = [] if r is None or not r["robot_visible"] else r["robot_ids"] + (r["mixed_ids"] if a.mixed == 1 else [])
            extra = []
            for rid in ([] if r is None or z is None or not r["robot_visible"] else r["mixed_ids"]):
                r2 = sel.get((name, t, rid))
                if r2 is not None and r2["robot_visible"] and f"lab_{t}_{rid}" in z.files:
                    extra.append((z[f"box_{t}_{rid}"], np.isin(z[f"lab_{t}_{rid}"], r2["robot_ids"])))
            zoomed += len(extra)
            masks.append(mask_from_ids(labs[t], ids, img.shape[:2], extra))
            used += r is not None
        if a.mixed == 2:
            # a region the VLM called part robot, part background is split by the robot's own motion: inside it, the
            # pixels that moved are robot. No extra API call; motion is trusted ONLY where the VLM saw robot.
            import cv2
            mv = motion_px(d["img"], masks)
            for t, img in enumerate(d["img"]):
                r = sel.get((os.path.basename(p), t, 0))
                if r is None or not r["robot_visible"] or not r["mixed_ids"]:
                    continue
                mixed = cv2.resize(np.isin(labs[t], r["mixed_ids"]).astype(np.uint8), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
                masks[t] = masks[t] | cv2.morphologyEx(mv[t] & mixed, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        if "mask" in d:
            d["mask_owl"] = d["mask"]
        d["mask"] = np.stack(masks).astype(np.uint8)
        np.savez_compressed(os.path.join(a.out_dir, os.path.basename(p)), **d)
        print(f"{os.path.basename(p)}: {used}/{len(masks)} frames answered, {zoomed} zoomed regions, robot pixel fraction {d['mask'].mean():.3f}"
              + (f" (was {d['mask_owl'].mean():.3f})" if "mask_owl" in d else ""), flush=True)


# ------------------------------------------------------------------ eval (mujoco)
def evaluate(a):
    import cv2
    import droid_selfcal as ds
    rob = ds.Franka(a.menagerie, hand=True)
    files = sorted(glob.glob(os.path.join(a.out_dir, "*.npz")))
    valid = [float(np.load(p, allow_pickle=True)["intr"][2]) for p in files]
    valid = [v for v in valid if np.isfinite(v) and v > 0]
    median_fy = float(np.median(valid)) if valid else 530.0
    rows = []
    for p in files:
        d = np.load(p, allow_pickle=True)
        if "mask_owl" not in d:
            continue
        fx, _, fy, _ = d["intr"]
        fy = fy if (np.isfinite(fy) and fy > 0) else (fx if (np.isfinite(fx) and fx > 0) else median_fy)
        wh_y = float(d["intr_wh"][1]) if (np.isfinite(d["intr_wh"][1]) and d["intr_wh"][1] > 0) else 720.0
        fy_px = float(fy) * rob.H / wh_y
        T = np.eye(4)
        T[:3, :3], T[:3, 3] = ds.euler_xyz_to_mat(d["ext"][3:6]), d["ext"][:3]
        Tg = T if a.convention == "cam2base" else np.linalg.inv(T)
        small = lambda ms: [cv2.resize(np.asarray(m, np.uint8), (rob.W, rob.H), interpolation=cv2.INTER_NEAREST) for m in ms]
        arms = {"owl": small(ds.motion_masks(d["img"], d["mask_owl"])), "vlm": small(ds.motion_masks(d["img"], d["mask"])),
                "vlm_nomotion": small([ds.clean_mask(m, keep_frac=0.15) for m in d["mask"]])}     # what calibrate --motion 0 --keep_frac 0.15 fits
        row = dict(file=os.path.basename(p), reliable=bool(float(d["quality"]) < 20))
        for k, ms in arms.items():
            pr, rc, io = [], [], []
            for i, m in enumerate(ms):
                ren = rob.mask(d["q"][i], Tg[:3, 3], Tg[:3, :3], fy_px)
                if ren.sum() < 20 or m.sum() == 0:            # the reference puts no robot in view, or no mask: not scored
                    continue
                pr.append((ren & m).sum() / m.sum()); rc.append((ren & m).sum() / ren.sum()); io.append(ds.iou(ren, m))
            row.update({f"{k}_precision": float(np.mean(pr)) if pr else None, f"{k}_recall": float(np.mean(rc)) if rc else None,
                        f"{k}_iou": float(np.mean(io)) if io else None, f"{k}_frames": len(pr)})
        rows.append(row)
    with open(os.path.join(a.out_dir, "mask_eval.jsonl"), "w") as f:
        f.writelines(json.dumps(r) + "\n" for r in rows)
    med = lambda rs, k: float(np.median([r[k] for r in rs if r[k] is not None])) if any(r[k] is not None for r in rs) else float("nan")
    print("masks against the robot rendered at DROID's reference pose (the reference is itself an estimate):")
    for name, rs in (("all cameras", rows), ("reliable reference", [r for r in rows if r["reliable"]])):
        print(f"\n{name}: {len(rs)}")
        print(f"  {'mask':<14}{'precision':>11}{'recall':>9}{'IoU':>8}{'frames scored':>15}")
        for k in ("owl", "vlm", "vlm_nomotion"):
            print(f"  {k:<14}{med(rs, k + '_precision'):>11.3f}{med(rs, k + '_recall'):>9.3f}{med(rs, k + '_iou'):>8.3f}"
                  f"{sum(r[k + '_frames'] for r in rs):>15}")
    better = sum(1 for r in rows if r["vlm_iou"] is not None and r["owl_iou"] is not None and r["vlm_iou"] > r["owl_iou"])
    both = sum(1 for r in rows if r["vlm_iou"] is not None and r["owl_iou"] is not None)
    print(f"\nIoU higher with the VLM masks on {better} of {both} cameras scored under both")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["propose", "select", "refine", "assemble", "eval"])
    ap.add_argument("--in_dir", default="")
    ap.add_argument("--work_dir", default="")
    ap.add_argument("--out_dir", default="")
    ap.add_argument("--frames", default="all", help="all, or frame indices such as 0,4,7")
    ap.add_argument("--limit", type=int, default=0, help="first N cameras only (0 = all)")
    ap.add_argument("--points_per_batch", type=int, default=64)
    ap.add_argument("--dump_marked", type=int, default=1, help="save one numbered frame per camera next to the proposals")
    ap.add_argument("--sam", default="", help="SAM checkpoint; default sam-vit-huge on GPU, sam-vit-base on CPU")
    ap.add_argument("--round", type=int, default=1, choices=[1, 2], help="select: 1 = whole frames, 2 = the zoomed crops from refine")
    ap.add_argument("--max_mixed", type=int, default=3, help="refine: at most this many mixed regions per frame")
    ap.add_argument("--model", default="claude-opus-5")
    ap.add_argument("--effort", default="medium", choices=["low", "medium", "high", "xhigh", "max"])
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--mixed", type=int, default=2, help="regions the VLM calls part robot, part background: 0 = leave out, "
                    "1 = take whole, 2 = take the pixels inside them that moved between frames (default)")
    ap.add_argument("--menagerie", default="/content/mujoco_menagerie")
    ap.add_argument("--convention", default="cam2base", choices=["cam2base", "base2cam"])
    a = ap.parse_args()
    a.work_dir = a.work_dir or os.path.join(a.in_dir or a.out_dir, "vlm_work")
    dict(propose=propose, select=select, refine=refine, assemble=assemble, eval=evaluate)[a.mode](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/droid_robotseg.py
"""BASELINE for the DROID self-calibration: robot masks from RobotSeg (Mei et al., CVPR 2026, github.com/showlab/RobotSeg),
a SAM 2 model trained to segment robots, in place of the masks our VLM picks out of a SAM partition. Same cameras, same
frames, same calibration search afterwards, so the only thing that changes is where the mask comes from.

Each camera file (<in_dir>/*.npz: img (T, H, W, 3) RGB, the T frames of one episode in time order) is written out as a
short JPEG video, RobotSeg runs in its AUTO mode (no click, no box: `add_new_robot` on frame 0, category `robot`), and the
file is copied to --out_dir with `mask` replaced (the old one is kept as `mask_owl`).

Run with the working directory at <RobotSeg>/test, because the repository resolves its config and checkpoint relative to it:
    cd /content/RobotSeg/test && python /content/scripts/droid_robotseg.py --in_dir ... --out_dir ... [--limit 2]
"""
import argparse
import glob
import os
import shutil
import tempfile

import numpy as np


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--limit", type=int, default=0, help="first N cameras only (0 = all): use 2 for the smoke test")
    ap.add_argument("--category", default="robot", choices=["robot", "arm", "gripper"])
    ap.add_argument("--yaml", default="robotseg-infer")
    ap.add_argument("--ckpt", default="../checkpoints/robotseg.pt")
    a = ap.parse_args()
    assert os.path.abspath(a.in_dir) != os.path.abspath(a.out_dir), "--out_dir must differ from --in_dir: the old masks are kept"
    assert os.path.exists(a.ckpt), f"RobotSeg checkpoint not found at {os.path.abspath(a.ckpt)}"
    os.makedirs(a.out_dir, exist_ok=True)

    import cv2
    import torch
    from robotseg.build_robotseg import build_robotseg_video_predictor
    predictor = build_robotseg_video_predictor(f"../robotseg/configs/{a.yaml}", a.ckpt)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    files = sorted(glob.glob(os.path.join(a.in_dir, "*.npz")))
    files = files[:a.limit] if a.limit else files
    for p in files:
        out = os.path.join(a.out_dir, os.path.basename(p))
        if os.path.exists(out):
            continue
        with np.load(p, allow_pickle=True) as z:
            d = dict(z)
        imgs = d["img"]
        tmp = tempfile.mkdtemp()
        try:
            for t, im in enumerate(imgs):
                cv2.imwrite(os.path.join(tmp, f"{t:05d}.jpg"), np.ascontiguousarray(im[..., ::-1]), [cv2.IMWRITE_JPEG_QUALITY, 97])
            masks = np.zeros(imgs.shape[:3], np.uint8)
            with torch.inference_mode(), torch.autocast("cuda", dtype=dtype):
                state = predictor.init_state(video_path=tmp, async_loading_frames=False, offload_video_to_cpu=False,
                                             offload_state_to_cpu=False)
                predictor.add_new_robot(inference_state=state, frame_idx=0, obj_id=0, robot=a.category)
                for t, ids, logits in predictor.propagate_in_video(inference_state=state, robot=a.category):
                    for i, oid in enumerate(ids):
                        if oid == 0:
                            m = np.squeeze((logits[i] > 0.0).detach().cpu().numpy()).astype(np.uint8)
                            if m.shape != imgs.shape[1:3]:
                                m = cv2.resize(m, (imgs.shape[2], imgs.shape[1]), interpolation=cv2.INTER_NEAREST)
                            masks[t] = m
        finally:
            shutil.rmtree(tmp, ignore_errors=True)
        if "mask" in d:
            d["mask_owl"] = d["mask"]
        d["mask"] = masks
        np.savez_compressed(out, **d)
        print(f"{os.path.basename(p)}: robot pixel fraction {masks.mean():.3f}"
              + (f" (zero-shot detector masks {d['mask_owl'].mean():.3f})" if "mask_owl" in d else "") + f", frames with a mask {int((masks.reshape(len(masks), -1).sum(1) > 0).sum())}/{len(masks)}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%bash
set -e
pip -q install "mujoco>=3.1" opencv-python-headless anthropic
# Colab's tensorflow_datasets links a tensorflow-metadata generated with a newer protobuf than the runtime; this pin is loadable
pip -q install --no-deps "tensorflow-metadata==1.16.1"
[ -d /content/mujoco_menagerie ] || git clone -q --depth 1 https://github.com/google-deepmind/mujoco_menagerie.git /content/mujoco_menagerie
ls /content/mujoco_menagerie/franka_emika_panda/panda_nohand.xml
python -c "import mujoco, tensorflow_datasets, transformers; print('mujoco', mujoco.__version__, '| transformers', transformers.__version__)"

In [ ]:
# 1. export (droid_100 from GCS, joined to the calibration files)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_selfcal.py export --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal --max_cameras 60 2>&1 | tee -a /content/logs/droid_selfcal.log

In [ ]:
# 2. segment the robot in the real frames
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_selfcal.py segment --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal 2>&1 | tee -a /content/logs/droid_selfcal.log

In [ ]:
# 3. self-calibrate every camera (resumable); the first line printed is the convention check
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl python /content/scripts/droid_selfcal.py calibrate --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal 2>&1 | tee -a /content/logs/droid_selfcal.log

In [ ]:
# 4. report (paste this)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_selfcal.py report --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal 2>&1 | tee -a /content/logs/droid_selfcal.log

In [ ]:
# VLM masks 1: SAM cuts every frame into numbered regions (~15 min on T4, resumable, no API)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_vlm_seg.py propose --in_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal --work_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal/vlm_work 2>&1 | tee -a /content/logs/droid_vlm_seg.log

In [ ]:
# VLM masks 2, PILOT: the VLM picks the robot regions on the first 6 cameras (Colab secret ANTHROPIC_API_KEY; about 2 dollars, prints the real cost), then masks old vs new against the reference render (paste the table)
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_vlm_seg.py select --in_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal --work_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal/vlm_work --limit 6 2>&1 | tee -a /content/logs/droid_vlm_seg.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_vlm_seg.py assemble --in_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal --work_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal/vlm_work --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_vlm_pilot --limit 6 2>&1 | tee -a /content/logs/droid_vlm_seg.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl python /content/scripts/droid_vlm_seg.py eval --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_vlm_pilot 2>&1 | tee -a /content/logs/droid_vlm_seg.log

In [ ]:
# VLM masks 3: every camera (resumable; answered frames are reused and never paid for twice). The zoom round is NOT run: the robot's own motion splits the part-robot regions for free
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_vlm_seg.py select --in_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal --work_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal/vlm_work 2>&1 | tee -a /content/logs/droid_vlm_seg.log

In [ ]:
# VLM masks 4: build the masks, score them against the reference render (paste the table)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_vlm_seg.py assemble --in_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal --work_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal/vlm_work --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_vlm 2>&1 | tee -a /content/logs/droid_vlm_seg.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl python /content/scripts/droid_vlm_seg.py eval --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_vlm 2>&1 | tee -a /content/logs/droid_vlm_seg.log

In [ ]:
# VLM masks 5: self-calibrate on the new masks (same v6 search, ~2 min per answered camera, resumable). Motion completion OFF (it is what re-attached the background) and every large mask component kept
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl python /content/scripts/droid_selfcal.py calibrate --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_vlm --convention cam2base --motion 0 --keep_frac 0.15 2>&1 | tee -a /content/logs/droid_selfcal.log

In [ ]:
# VLM masks 6: report (paste this)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_selfcal.py report --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_vlm 2>&1 | tee -a /content/logs/droid_selfcal.log

In [ ]:
# RobotSeg baseline 1: install RobotSeg (CVPR 2026, a SAM 2 model trained to segment robots) and find its checkpoint (~6 min). The authors host the weights on OneDrive only, which a notebook cannot fetch: download robotseg.pt ONCE from the link in github.com/showlab/RobotSeg (Checkpoint section) and put it on Drive at restoring_organs/plus/robotseg/robotseg.pt
import os
CK = '/content/drive/MyDrive/restoring_organs/plus/robotseg/robotseg.pt'
assert os.path.exists(CK), 'put robotseg.pt on Drive at ' + CK + ' (link: github.com/showlab/RobotSeg, Checkpoint section), then re-run this cell'
assert os.path.getsize(CK) > 100e6, 'robotseg.pt on Drive is only ' + str(os.path.getsize(CK) // 1000000) + ' MB: the download was cut short (the model is a SAM 2.1 network, well over 100 MB). Download it again, wait until the browser says it is complete, replace the file on Drive, then re-run'
![ -d /content/RobotSeg ] || git clone -q --depth 1 https://github.com/showlab/RobotSeg /content/RobotSeg
!cd /content/RobotSeg && pip install -q --no-deps -e . && pip install -q hydra-core iopath natsort omegaconf kornia && (python setup.py build_ext --inplace > /content/robotseg_build.log 2>&1 || echo 'optional CUDA extension did not build (only post-processing uses it): continuing')
!mkdir -p /content/RobotSeg/checkpoints && cp -f /content/drive/MyDrive/restoring_organs/plus/robotseg/robotseg.pt /content/RobotSeg/checkpoints/ && ls -la /content/RobotSeg/checkpoints

In [ ]:
# RobotSeg baseline 2, SMOKE: two cameras only (~1 min). Paste the two lines it prints: a robot pixel fraction between about 0.02 and 0.30 means it works; an error here costs nothing, send it to the agent
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/RobotSeg/test && python /content/scripts/droid_robotseg.py --in_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal --out_dir /content/robotseg_smoke --limit 2 2>&1 | tee -a /content/logs/droid_robotseg.log

In [ ]:
# RobotSeg baseline 3: every camera (~10 min on T4, resumable), then masks scored against the reference render (paste the table)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/RobotSeg/test && python /content/scripts/droid_robotseg.py --in_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_robotseg 2>&1 | tee -a /content/logs/droid_robotseg.log
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl python /content/scripts/droid_vlm_seg.py eval --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_robotseg 2>&1 | tee -a /content/logs/droid_vlm_seg.log

In [ ]:
# RobotSeg baseline 4: the SAME self-calibration search on RobotSeg's masks (~2 min per camera, ~2 h, resumable), same settings as the VLM masks
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; MUJOCO_GL=egl python /content/scripts/droid_selfcal.py calibrate --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_robotseg --convention cam2base --motion 0 --keep_frac 0.15 2>&1 | tee -a /content/logs/droid_selfcal.log

In [ ]:
# RobotSeg baseline 5: report (paste this)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; python /content/scripts/droid_selfcal.py report --out_dir /content/drive/MyDrive/restoring_organs/plus/droid_selfcal_robotseg 2>&1 | tee -a /content/logs/droid_selfcal.log